# Expected Pass Value (EPV)
Expected Pass Value (EPV) is a metric used to assign a value to a pass played at a given location. In this notebook, we will walk through a simple model to calculate the EPV of a pass, so we can quantify how a pass played impacts our likelihood of scoring a goal. We will then expand on this concept and use EPV to evaluate a player's decision-making.

---

**Change Directory**

This block of code is only ran so that, so this notebook can use the FMA library.

In [ ]:
from pathlib import Path
import sys
import os

cwd = os.getcwd()
os.chdir(cwd)

project_root = Path.cwd().parent
sys.path.append(str(project_root))

**Import FMA Library**

In [ ]:
from FootballMatchAnalysis.objects.match import Match
from FootballMatchAnalysis.analysis.xt import *

**Load Event and Tracking Data**

In [ ]:
DATADIR = './../data'
game_id = 1

In [ ]:
match = Match(DATADIR, game_id)

**Reviewing a Moment**

We will start by examining frame 27942 of the match, where Player 5 passes to Player 8. Let's determine how this pass has affected the chances of scoring.

In [ ]:
# Get Moment
moment = match.get_moment(27942)
moment.event

In [ ]:
plot = moment.plot_moment(include_player_velocities=True)

**Expected Threat (xT) at The Current Location**

The first step to quantify the value of Player 5 having the ball in their current location. To do so, we'll use Expected Threat (xT). xT quantifies our chances of scoring when possessing the ball at a given location. 

In [ ]:
# Get Current Ball Location
ball = moment.ball
start = (ball.x, ball.y)

# Get xT at Ball Location
start_xt = get_xt(start)
print(f"The xT at {start} is {start_xt}")

**Expected Threat (xT) at the Pass's Final Destination**

Our next step is to quantify the value of a pass being received at its final destination. We will then subtract the `end_xt` from the `start_xt` to understand how this pass changes the xT.

In [ ]:
## Get Pass's Final Destination
# Pass played
p = moment.event
target = (p["End X"], p["End Y"])

# Get xT at Final Destination
end_xt = get_xt(target)
xt_gained = end_xt - start_xt

print(f"The xT at {target} is {end_xt}")
print(f"Our change in xT is {xt_gained}")

**Pass Probability**

We must now consider the probability of a pass being completed.

In [ ]:
# Probability of a Completed Pass
prob = moment.pass_probability(target)
print(f"The P(Success) is {round(prob * 100, 2)}%")

**What If Our Pass Fails?**

In order to evaluate the full value of a pass, we must also consider what happens if the pass fails. For this reason, we need to calculate the xT that the opponent would receive from intercepting the ball at the pass's final destination, how that changes from our current xT and the probability of a pass failing.

In [ ]:
# xT of Opponent, If They Intercept the Pass 
opp_end_xt = end_xt = get_xt(target, invert=True)    

# Change in xT If Pass is Intercepted
xt_lossed = (-1 * opp_end_xt) - start_xt

# Probability of a Failed Pass
prob_failure = 1 - prob

print(f"Opponent xT: {opp_end_xt}")
print(f"Opponent Change in xT: {xt_lossed}")
print(f"The P(Failure) is {round(prob_failure * 100, 2)}%")

**Calculating EPV**

We are now ready to calculate EPV using all the pieces we previously calculated. Our formula for EPV is...

`EPV = ( P(Success) * △ xT ) + ( (1 - P(Success)) * Opp △ xT )`

In [ ]:
#EPV Calculuation
epv = (prob * xt_gained) + (prob_failure * xt_lossed)

print(f"The EPV of Player 5's pass to Player 8 is {round(epv, 5)}")
print(f"This means that this pass increases our likelyhood of scoring by {round(epv*100, 3)}%")

**Functionalizing EPV**

To simplify the calculation and use of EPV in our match analysis, we are now going to take our calculations above and compile them into a single function. `calculate_epv` will take a Moment object and a given target and calculate the EPV of a pass played to that location for that given Moment.

In [ ]:
def calculate_epv(moment, target):
    # Get Start and End of Pass
    ball = moment.ball
    start = (ball.x, ball.y)

    # Get xT and xT Generated From The Pass
    start_xt = get_xt(start)
    end_xt = get_xt(target)
    xt_gained = end_xt - start_xt

    # Probability of a Completed Pass
    prob = moment.pass_probability(target)
    # Probability of a Failed Pass
    prob_failure = 1 - prob

    # xT of Opponent, If They Intercept the Pass 
    opp_end_xt = end_xt = get_xt(target, invert=True)

    # Change in xT If Pass is Intercepted
    xt_lossed = (-1 * opp_end_xt) - start_xt

    # Final EPV
    epv = (prob * xt_gained) + (prob_failure * xt_lossed)
    return epv

**Was This The Right Pass Played?**

Looking at the moment, Player 5 was on the break and actually had multiple passing options. In addition to player 8, Player 5 could have played a pass to Players 6, 9 or 10 as well. Let's now use EPV to evaluate passed to those players and determine if Player 5 made the right decision.

In [ ]:
plot = moment.plot_moment(include_player_velocities=True)
plot = moment.plot_pitch_control(plot=plot)
plot

In [ ]:
# Getting the Players We Want to Review
home_team = moment.home_team()
player_6 = home_team[10]
player_9 = home_team[13]
player_10  = home_team[0]
players = [player_10, player_9, player_6]

**Find the Best Passing Location for Each Player**

To evaluate which passing option is the best, we must first evaluate the optimal pass played to each player. For-example, Player 5 does not pass directly to Player 8, he passes in front of Player 8 so Player 8 can run onto the ball.

To find the optimal pass to play to a player, we will use a series of Player helper functions within the FMA library to scan points around each player and identify the point that yields the highest EPV.

In [ ]:
# Use to Save the Optimal Pass
passes = {}

# Review Passing Options
for player in players:
    best_epv, best_pass = None, None

    ## Scan Around The Player
    # We will use a player's velocity as a radius 
    # and scan the circle around that player for 
    # locations to potentially pass to. Points are
    # generated by `Player.coords_in_radius`.
    for target in player.coords_in_radius(radius=player.speed):
            
            ## Make Sure the Pass Is In Front
            # To make sure the pass is in front of the player
            # we'll use `Player.in_direct_view` to ensure the
            # potential pass location is within 45 degrees of 
            # the player's velocity.
            if player.in_direct_view(target[0], target[1]):

                #Calculate EPV
                pot_epv = calculate_epv(moment, target)

                # Save Results If It's the Best EPV
                if best_epv is None or pot_epv > best_epv:
                    best_epv = pot_epv
                    best_pass = target
                    
    # Save Results
    passes[f"Player {player.name}"] = { "EPV" : best_epv, "Pass" : best_pass}

**Review The Results**

Reviewing the results, it turns out the right pass was made. A pass to Player 8 narrowly edges out a pass to Player 10 by .001019.

This is of course a marginal difference, but you can see how simple this model was to run and evaluate a player's decision making for a single moment. You could expand this to every passing moment, aggregate the results and quantify who the best decision maker on the pitch is.

In [ ]:
for player in passes:
    print(f"A Pass to {player} yields an EPV of {round(passes[player]["EPV"], 6)}")

print(f"A Pass to Player 8 yields an EPV of {round(epv, 6)}")